In [12]:

"""
Project: Trading Bot Using Indicators of choice

Goal:
The goal of this project is to build a trading bot that will show good opportunites to buy stock based on certain indicators. The indicators
chosen are : Stochastic RSI, RSI (Relative Strength Index), Moving Average (200 day). These will be viewed from the weekly chart. 
With this tool we would be able to scrape stocks for potential setups that mirror what I would look for in a trade setup.
The strategy will then be backtested to show its success rate. The trading bot itself will then point out if a stock is 
a "buy" depending on whether indicators are showing the correct parameters

Features:
- Stock indicator check
- Backtesting tool
- Multi-stock scanner
- SQL Integration to store database of historical data (querying for backtesting)

Status: Working on inputing the strategy (based on my actual trading strategy)
"""

import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime
import sqlite3

In [2]:
# Strategy Settings

'''
Strategy Rules (all of these are on the weekly timeframe):

Buy the stock if:
- RSI > 30
- StochRSI < 20 (both K and D lines)
- price is > 200 day moving average

Otherwise, the stock is not a Buy
'''
# Thresholds for the indicators

RSI_Threshold = 30
SMA_Period = 200
StochRSI_Threshold = 20

In [3]:
# Indicator Functions

#RSI Formula uses Wilder smoothing, math to make it similar to TradingView's RSI
def calculate_rsi(close, period=14):
    delta = close.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    return rsi

# K and D periods for RSI both resemble those of TradingView
def calculate_stoch_rsi(close, rsi_period=14, stoch_period=14, k_period=3, d_period=3):
    rsi = calculate_rsi(close, rsi_period)

    min_rsi = rsi.rolling(stoch_period).min()
    max_rsi = rsi.rolling(stoch_period).max()

    stoch_rsi = (rsi - min_rsi) / (max_rsi - min_rsi)

    k = stoch_rsi.rolling(k_period).mean()
    d = k.rolling(d_period).mean()

    return {'k': k, 'd': d}


def calculate_sma(close, period=200):
    sma = close.rolling(window=period).mean()

    return sma
    

In [4]:
# Analyze Indicator Functions

def analyze_rsi(last_rsi):

    if pd.isna(last_rsi):
        return 'No RSI data available'
        
    if last_rsi < 20:
        return 'This stock is way oversold'
    elif last_rsi > 20 and last_rsi < 40:
        return 'Stock is leaning oversold'
    elif last_rsi > 40 and last_rsi < 60:
        return 'RSI is near midpoint range'
    elif last_rsi > 60 and last_rsi < 70:
        return 'Stock is leaning overbought'
    elif last_rsi >= 70:
        return 'This stock is overbought for sure'


def analyze_stoch(stoch_k, stoch_d):

    if pd.isna(stoch_k) or pd.isna(stoch_d):
        return 'No StochRSI data available'
        
    if stoch_k <= .20 and stoch_d <= .20:
        return 'Momentum is bearish'
    elif stoch_k >= .70 and stoch_d >= .70:
        return 'Full bullish momentum'
    elif stoch_k >= .70 and stoch_d <= .70: # k line can lead before d fully confirms the bullishness
        return 'StochRSI is showing bullish momentum'
    else:
        return 'StochRSI in midrange, evaluate further'

def analyze_sma(close, sma_value):

    if pd.isna(sma_value): # edge case to check if there is any value in there.
        return 'No SMA(200) data available'

    if close.iloc[-1] >= sma_value:
        return 'This stock is bull market bullish'
    else:
        return 'Stock may be in a bear trend'



In [5]:
# Multi-Stock Scanner

def analyze_ticker(ticker):

    timestamp = datetime.now() #setting the timestamp, this can be used for the databse.
    
    df = yf.download(ticker, period='5y', interval='1wk') # pulling data from yfinance
    
    close = df['Close'][ticker] # Extracting closing price series

    # ---- RSI ----
    rsi = calculate_rsi(close, 14)
    last_rsi = rsi.iloc[-1]
    rsi_analysis = analyze_rsi(last_rsi)

    # ---- StochRSI ----
    stoch_rsi = calculate_stoch_rsi(close)
    stoch_k = stoch_rsi['k'].iloc[-1]
    stoch_d = stoch_rsi['d'].iloc[-1]
    stoch_analysis = analyze_stoch(stoch_k, stoch_d)

    # ---- SMA(200) ----
    sma_real = calculate_sma(close)
    sma_value = sma_real.iloc[-1]
    sma_analysis = analyze_sma(close, sma_value)

    # ---- Creating the row ----
    return {
    'Timestamp': timestamp,
    'Ticker': ticker,
    'Price': round(close.iloc[-1], 2),
    'RSI': round(last_rsi, 2),
    'RSI_Comment': rsi_analysis,
    'Stoch_K': round(stoch_k, 2),
    'Stoch_D': round(stoch_d, 2),
    'StochRSI_Comment': stoch_analysis,
    'SMA(200)': round(sma_value, 2)  if not pd.isna(sma_value) else 'N/A',
    'SMA_Comment': sma_analysis    
    }

# calling the functions and creating the table

tickers = ['AAPL', 'MSFT', 'META', 'CRM', 'IBIT', 'NVDA', '131']

def run_scanner(tickers):

    rows = []
    
    for ticker in tickers:

        try:
            result = analyze_ticker(ticker)
            rows.append(result)
        except Exception as e:
            rows.append({
                'Ticker': ticker,
                'Price': 'N/A',
                'RSI': 'N/A',
                'RSI_Comment': 'Invalid ticker',
                'Stoch_K': 'N/A',
                'Stoch_D': 'N/A',
                'StochRSI_Comment': 'Invalid ticker',
                'SMA(200)': 'N/A',
                'SMA_Comment': 'Invalid ticker'
            })

    scanner_table = pd.DataFrame(rows)

    return scanner_table

scanner_table = run_scanner(tickers)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$131: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['131']: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")


In [6]:
# Bullish Scores
# Using the indicator values from the rows themselves..

def momentum_score(row):
    score = 0

    if ( 
        row['RSI'] == 'N/A'
        or row['Price'] == 'N/A'
        or row['Stoch_K'] == 'N/A'
        or row['Stoch_D'] == 'N/A'
        or row['SMA(200)'] == 'N/A'
    ):
        return 'Not enough data' # This checks for missing data if not then function throws error for 'N/A'

    #RSI Scorer
    if row['RSI'] >= 70:
        score += 2
    elif row['RSI'] >= 60:
        score += 1
    elif row['RSI'] >= 50:
        score += 0.5
    elif row['RSI'] <= 50:
        score += -0.5
    elif row['RSI'] <= 40:
        score += -1
    elif row['RSI'] <= 20:
        score += -2
    
    #StochRSI Scorer
    if row['Stoch_K'] >= .70 and row['Stoch_D'] >= .70:
        score += 2
    elif row['Stoch_K'] >= 70 or row['Stoch_D'] >= .70:
        score += 1
    elif row['Stoch_K'] >= .50 or row['Stoch_D'] >= .50:
        score += .5
    elif row['Stoch_K'] >= .20 or row['Stoch_D'] >= .20:
        score += .25

    #SMA(200) Scorer
    if row['Price'] > row['SMA(200)']:
        score += 2
    elif row['Price'] < row['SMA(200)']:
        score += -2

    return score


# Adding a new column in scanner_table 'Momentum Score', applying the momentum_score function to get the values
scanner_table['Momentum_Score'] = scanner_table.apply(momentum_score, axis=1)

scanner_table

,Timestamp,Ticker,Price,RSI,RSI_Comment,Stoch_K,Stoch_D,StochRSI_Comment,SMA(200),SMA_Comment,Momentum_Score
0,2026-06-28 16:26:18.734326,AAPL,283.78,53.46,RSI is near midpoint range,0.43,0.62,"StochRSI in midrange, evaluate further",206.14,This stock is bull market bullish,3.0
1,2026-06-28 16:26:19.092685,MSFT,372.97,38.43,Stock is leaning oversold,0.46,0.59,"StochRSI in midrange, evaluate further",381.81,Stock may be in a bear trend,-2.0
2,2026-06-28 16:26:19.220985,META,550.25,40.1,RSI is near midpoint range,0.38,0.46,"StochRSI in midrange, evaluate further",464.35,This stock is bull market bullish,1.75
3,2026-06-28 16:26:19.329668,CRM,158.37,36.86,Stock is leaning oversold,0.34,0.53,"StochRSI in midrange, evaluate further",232.21,Stock may be in a bear trend,-2.0
4,2026-06-28 16:26:19.436471,IBIT,33.85,32.78,Stock is leaning oversold,0.19,0.22,"StochRSI in midrange, evaluate further",N/A,No SMA(200) data available,Not enough data
5,2026-06-28 16:26:19.543126,NVDA,192.53,48.88,RSI is near midpoint range,0.48,0.54,"StochRSI in midrange, evaluate further",103.72,This stock is bull market bullish,2.0
6,NaT,131,N/A,N/A,Invalid ticker,N/A,N/A,Invalid ticker,N/A,Invalid ticker,Not enough data


In [7]:
# testing with StochRSI. We are working on getting the historical data for stochrsi to show where it came from. 
# if StochRSI came from above the 70 level and crossed downward, that is a bearish sign for momentum.
# We want to capture this move as it will determine the real momentum of the stock vs just using static levels of StochRSI

#stoch_rsi = calculate_stoch_rsi(close)


# looking at the analyze_stoch function. since it does not take all historical values yet. We may have to change this to then take data for historical data
# We need to atleast see the last several data points of data to determine the change of the StochRSI.


def analyze_stoch(stoch_k, stoch_d):

    # This shows the difference between the last 5 stoch_K and stoch_d values, and gets the average of that difference.
    recent_k_diff = stoch_k.tail(5).diff().mean()
    recent_d_diff = stoch_d.tail(5).diff().mean()

    # Interprets the change in stochRSI K and D values mean difference
    if recent_k_diff < 0 and recent_d_diff < 0:
        return 'StochRSI is losing momentum!'
    elif recent_k_diff > 0 and recent_d_diff > 0:
        return 'StochRSI is gaining momentum!'

    # And what about the K and D values? What do they say?
    if last_stoch_k < last_stoch_d:
        return 'StochRSI K line is below D line. Bullish momentum may be fading'

    # For simplicity we calculate the last values of Stoch K and D
    last_stoch_k = stoch_k['k'].iloc[-1]
    last_stoch_d = stoch_d['d'].iloc[-1]
    
    if pd.isna(last_stoch_k) or pd.isna(last_stoch_d):
        return 'No StochRSI data available'
        
    if last_stoch_k <= .20 and last_stoch_d  <= .20:
        return 'Momentum is bearish'
    elif last_stoch_k >= .70 and last_stoch_d >= .70:
        return 'Full bullish momentum'
    elif last_stoch_k >= .70 and last_stoch_d <= .70: # k line can lead before d fully confirms the bullishness
        return 'StochRSI is showing bullish momentum'
    else:
        return 'StochRSI in midrange, evaluate further'

    

    # ---- StochRSI ---- as appears in the analyze_ticker function, we should change both stoch_k and stoch_d to be not iloc.

    # stoch_rsi = calculate_stoch_rsi(close)
    # stoch_k = stoch_rsi['k']
    # stoch_d = stoch_rsi['d']
    # stoch_analysis = analyze_stoch(stoch_k, stoch_d)


In [27]:
# SQL integration
# This is going to build a database to then store the results we get from the indicators. We will then query this to get historical data

conn = sqlite3.connect('trading_bot.db') # setting a connection for SQL

scanner_table.to_sql('scanner_results',conn,if_exists='replace',index=False) # adding our results from scanner_table into the trading_bot database, replace if table already exists

pd.read_sql(
    "SELECT * FROM scanner_results", conn)

pd.read_sql(
    'SELECT * FROM scanner_results WHERE ticker = "AAPL"', conn)

# Testing out using queries, but the database does work for sure. 
AAPL = '''
SELECT *
FROM scanner_results
WHERE ticker = "AAPL"'''

pd.read_sql(AAPL, conn)

,Timestamp,Ticker,Price,RSI,RSI_Comment,Stoch_K,Stoch_D,StochRSI_Comment,SMA(200),SMA_Comment,Momentum_Score
0,2026-06-28 16:26:18.734326,AAPL,283.78,53.46,RSI is near midpoint range,0.43,0.62,"StochRSI in midrange, evaluate further",206.14,This stock is bull market bullish,3.0


In [8]:
# Some test scripts here

# Notes
# Backtester
# Test: “When this setup appeared, what was the 3-month return?”

In [9]:
# Signal Ticker Test

In [10]:
# Backtester

In [11]:
# Results / Charts